# Movie features

Turn the cleaned movie table (`movies_features-raw.csv`, output of `03_cleaning_moviesData.ipynb`) into the static, model-ready `movies_features.csv`, applying the encoding decisions of `05_eda_movies_raw.ipynb` (section 12).

Only TMDB metadata is used here. Features computed from ratings (director / cast track records, movie `rating_std`, user statistics) depend on the train/test split and are built after it, in `07_split_and_user_eda.ipynb`, from training movies only.

## 1. Load

List columns are stored as text (e.g. `"['Drama', 'Comedy']"`) and are parsed back into Python lists.

In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED = Path("../data/processed")
LIST_COLS = ["genre_names", "company_names", "country_names", "language_names", "cast_names"]

movies = pd.read_csv(
    PROCESSED / "movies_features-raw.csv",
    converters={c: ast.literal_eval for c in LIST_COLS},
)
print(movies.shape)
movies.head(3)

## 2. Drop impossible budget / revenue

A handful of movies have budget or revenue below \$1,000 (e.g. *Modern Times* with a budget of \$1). These are data-entry errors (values in millions or missing), not real figures, and would distort `log_roi`.

In [ ]:
MIN_MONEY = 1_000

bad = (movies["budget"] < MIN_MONEY) | (movies["revenue"] < MIN_MONEY)
display(movies.loc[bad, ["netflix_movie_id", "title", "year", "budget", "revenue"]])
movies = movies[~bad].reset_index(drop=True)
print(f"dropped {int(bad.sum())} movies; remaining {len(movies)}")

## 3. Numeric features

- `runtime`: kept raw, no cap (tree models are not affected by the few very long films).
- `movie_age_log = log1p(2005 - year)`: age at the end of the Netflix data; `log1p` because 2005 films have age 0 and the tail of old classics is long.
- `log_roi = log(revenue / budget)`: box-office success, symmetric on the log scale.
- `budget_log`: kept for the persona profiles ("budget scale", RQ1) but not as a model feature, since `log_roi` already carries the budget/revenue information.

In [ ]:
movies["movie_age_log"] = np.log1p(2005 - movies["year"])
movies["log_roi"] = np.log(movies["revenue"]) - np.log(movies["budget"])
movies["budget_log"] = np.log(movies["budget"])

movies[["runtime", "movie_age_log", "log_roi", "budget_log"]].describe().round(2)

## 4. Genres

Multi-hot: one column per genre, no reference dropped (a movie has 1–7 genres). `Foreign` (1 movie) is dropped. `Documentary` is kept although it has only 10 movies: they carry ~370k ratings from ~42% of users, so documentary taste is measurable for persona clustering, and 5 movies have it as their only genre.

In [ ]:
DROP_GENRES = {"Foreign"}


def slugify(name):
    return "".join(ch if ch.isalnum() else "_" for ch in name.lower()).strip("_")


genres = sorted({g for gs in movies["genre_names"] for g in gs} - DROP_GENRES)
genre_df = pd.DataFrame(
    {f"genre_{slugify(g)}": movies["genre_names"].map(lambda gs, g=g: int(g in gs)) for g in genres}
)
print(f"{len(genres)} genre columns; movies with no kept genre: {int((genre_df.sum(axis=1) == 0).sum())}")
genre_df.sum().sort_values(ascending=False)

## 5. Production companies

Merge true aliases of the same studio (list from `05_eda_movies_raw` section 6), then keep the **top 15** companies as one-hot columns (each has ≥ 25 movies) and flag everything else in `company_other`.

In [ ]:
COMPANY_ALIASES = {
    "Columbia Pictures Corporation": "Columbia Pictures",
    "Paramount": "Paramount Pictures",
    "Warner Bros. Pictures": "Warner Bros.",
    "Universal Studios": "Universal Pictures",
    "Universal Pictures Corporation": "Universal Pictures",
    "Miramax": "Miramax Films",
    "Walt Disney": "Walt Disney Pictures",
    "Walt Disney Productions": "Walt Disney Pictures",
    "DreamWorks": "DreamWorks SKG",
    "DreamWorks Pictures": "DreamWorks SKG",
    "Polygram Filmed Entertainment": "PolyGram Filmed Entertainment",
    "Lions Gate": "Lions Gate Films",
    "Malpaso Company": "Malpaso Productions",
    "Cruise-Wagner Productions": "Cruise/Wagner Productions",
    "Mandalay Pictures": "Mandalay Entertainment",
    "Zanuck Company, The": "The Zanuck Company",
    "Kennedy/Marshall Company, The": "The Kennedy/Marshall Company",
}
TOP_COMPANIES = 15

# dict.fromkeys keeps the order and drops a duplicate when a movie listed both aliases
movies["company_names"] = movies["company_names"].map(
    lambda cs: list(dict.fromkeys(COMPANY_ALIASES.get(c, c) for c in cs))
)
company_counts = movies["company_names"].explode().dropna().value_counts()
top_companies = list(company_counts.index[:TOP_COMPANIES])

company_df = pd.DataFrame(
    {f"company_{slugify(c)}": movies["company_names"].map(lambda cs, c=c: int(c in cs)) for c in top_companies}
)
company_df["company_other"] = movies["company_names"].map(
    lambda cs: int(len(cs) == 0 or any(c not in top_companies for c in cs))
)
company_df.sum().sort_values(ascending=False)

## 6. Production countries

The US appears in 93% of movies, so instead of a top-X one-hot three groups are used: `us_only`, `us_coproduction` (US plus another country) and `non_us`. Movies with no listed country get 0 in all three.

In [ ]:
US = "United States of America"

country_df = pd.DataFrame({
    "us_only": movies["country_names"].map(lambda cs: int(cs == [US])),
    "us_coproduction": movies["country_names"].map(lambda cs: int(US in cs and len(cs) > 1)),
    "non_us": movies["country_names"].map(lambda cs: int(len(cs) > 0 and US not in cs)),
})
print("movies with no country:", int((country_df.sum(axis=1) == 0).sum()))
country_df.sum()

## 7. Language

English is the original language of 97.5% of movies, so a single `lang_en` flag replaces a per-language one-hot. `is_multilingual` flags films with 2+ spoken languages.

In [ ]:
language_df = pd.DataFrame({
    "lang_en": (movies["original_language"] == "en").astype(int),
    "is_multilingual": (movies["language_names"].map(len) > 1).astype(int),
})
language_df.mean().round(3)

## 8. People for the track records

Director and actor names are too sparse for one-hot columns (60% of directors and 74% of actors appear in one movie only). They are kept as names here and turned into `director_track_record` and `cast_track_record` after the split. `cast_top5` is the first 5 billed actors (`cast_names` is stored in billing order).

In [ ]:
CAST_K = 5

movies["cast_top5"] = movies["cast_names"].map(lambda names: names[:CAST_K])
print("movies without director:", int(movies["director"].isna().sum()))
print("cast_top5 length:", movies["cast_top5"].map(len).value_counts().sort_index().to_dict())

## 9. Assemble and save

`year` is kept for the decade stratification of the movie split, not as a model feature (`movie_age_log` replaces it).

In [ ]:
BASE_COLS = [
    "netflix_movie_id",
    "title",
    "year",
    "runtime",
    "movie_age_log",
    "log_roi",
    "budget_log",
]
PEOPLE_COLS = ["director", "cast_top5"]

features = pd.concat(
    [movies[BASE_COLS], genre_df, company_df, country_df, language_df, movies[PEOPLE_COLS]],
    axis=1,
)

assert features["netflix_movie_id"].is_unique
assert features.drop(columns=PEOPLE_COLS).notna().all().all()

out_path = PROCESSED / "movies_features.csv"
features.to_csv(out_path, index=False)
print(f"saved {out_path} ({features.shape[0]} rows x {features.shape[1]} cols)")
print(
    f"genres: {genre_df.shape[1]} | companies: {company_df.shape[1]} | "
    f"countries: {country_df.shape[1]} | language: {language_df.shape[1]}"
)
features.head(3)